# Dataset Size vs Cost

The goal of this notebook is to explore the difference in cost with increasing dataset sizes.

In [9]:
import pandas as pd
import numpy as np

import sys
sys.path.append("..")
from src.ingestor import download_and_clean_ascincome
from src.he_pipeline import (DataOwner, UntrustedResearcher, Measure, compare,
                         features_plain, package_nbytes)

In [6]:
df = download_and_clean_ascincome(states=['MA'], 
                             year=2018, 
                             horizon='1-Year', 
                             features='all', 
                             min_age=18)

States: MA
Survey: 2018 ACS 1-Year
Rows downloaded: 70,131
Rows after target/age filtering: 57,513
Rows removed: 12,618

Income-label counts:
income_over_50k
0    36841
1    20672
Name: count, dtype: int64


In [7]:
df_processed = df[["AGEP", "COW", "SCHL", "MAR", "RELP", "WKHP", "SEX", "RAC1P", "PINCP", "PWGTP", "income_over_50k"]]
df_processed = pd.get_dummies(df_processed, columns=["COW", "SCHL", "MAR", "RELP", "SEX", "RAC1P"], dtype=int)
df_processed = df_processed.dropna()
len(df_processed)

57513

In [8]:
# create samples of different sizes
sample_1k = df_processed.sample(n=1000, random_state=42)
sample_2k = df_processed.sample(n=2000, random_state=42)
sample_5k = df_processed.sample(n=5000, random_state=42)
sample_10k = df_processed.sample(n=10000, random_state=42)
sample_20k = df_processed.sample(n=20000, random_state=42)

## Run Experiment

In [11]:
samples = {
    "1k": df.loc[sample_1k.index],
    "2k": df.loc[sample_2k.index],
    "5k": df.loc[sample_5k.index],
    "10k": df.loc[sample_10k.index],
    "20k": df.loc[sample_20k.index]
}

costs = {}

## Generating features on raw data

In [12]:
# Measure generating the features on raw data
for key, sample in samples.items():
    with Measure(f'raw_features_{key}') as m:
        features_raw = features_plain(sample)
    costs.update(m.summary())

## Generating Features HE Pipeline

In [19]:
for key, sample in samples.items():
    # Data Owner
    with Measure(f'he_encryption_{key}') as m:
        owner = DataOwner()
        encoded = owner.encode(sample)
        package = owner.encrypt(encoded)
        public_context = owner.public_context()
    costs.update(m.summary())
    # Researcher
    with Measure(f"compute_features_{key}") as m:
        researcher = UntrustedResearcher(public_context)
        encrypted_stats = researcher.compute(package)
    costs.update(m.summary())
    # Data Owner
    with Measure(f'he_decrypt_{key}') as m:
        decrypted_stats = owner.decrypt(encrypted_stats)
        features_he = owner.finalize(decrypted_stats, package["n_rows"])
    costs.update(m.summary())
    print(f'Sample: {key}')
    print(f"Encrypted statistics returned: {len(encrypted_stats)}")
    print(f"Size of returned ciphertexts: {sum(map(len, encrypted_stats.values())) / 2**20:,.1f} MiB")

Sample: 1k
Encrypted statistics returned: 209
Size of returned ciphertexts: 52.0 MiB
Sample: 2k
Encrypted statistics returned: 209
Size of returned ciphertexts: 52.0 MiB
Sample: 5k
Encrypted statistics returned: 209
Size of returned ciphertexts: 52.0 MiB
Sample: 10k
Encrypted statistics returned: 209
Size of returned ciphertexts: 52.0 MiB
Sample: 20k
Encrypted statistics returned: 209
Size of returned ciphertexts: 52.0 MiB


In [20]:
cost_table = pd.DataFrame.from_dict(costs, orient="index")
cost_table

,latency_s,peak_rss_mib,peak_rss_increase_mib
raw_features_1k,0.019099,447.218750,0.156250
raw_features_2k,0.014202,447.265625,0.062500
raw_features_5k,0.013074,447.281250,0.046875
raw_features_10k,0.012370,447.328125,0.078125
raw_features_20k,0.014443,447.359375,0.078125
he_encryption_1k,0.335753,1333.375000,20.500000
he_encryption_2k,0.326357,1333.953125,0.203125
he_encryption_5k,0.502593,1334.671875,0.203125
he_encryption_10k,0.628263,1336.140625,1.593750
he_encryption_20k,0.930604,1342.593750,1.703125
